Importing packages

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from pandas.api.types import infer_dtype

/Users/piotr/UEP-MachineLearning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
user_path = "/Users/piotr/"

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [4]:
def embed_column(df: pd.DataFrame, col: str, model: SentenceTransformer) -> pd.DataFrame:
    texts = df[col].fillna("").astype(str).tolist()
    emb = model.encode(texts, show_progress_bar=True)
    df[f"{col}_emb"] = list(emb)
    return df

In [5]:
def explode_embeddings(df: pd.DataFrame, emb_cols: list[str]) -> pd.DataFrame:
    for col in emb_cols:
        if col not in df.columns:
            continue
            
        non_null = df[col].dropna()
        if non_null.empty:
            continue
            
        first_valid = non_null.iloc[0]
        emb_dim = len(first_valid)

        # Use apply instead of fillna
        def _ensure_array(x):
            if isinstance(x, (np.ndarray, list)) and x is not None:
                return x
            return np.zeros(emb_dim, dtype=float)
        
        emb_series = df[col].apply(_ensure_array)
        emb_matrix = np.vstack(emb_series.values)

        for i in range(emb_dim):
            df[f"{col}_{i}"] = emb_matrix[:, i]

        df.drop(columns=[col], inplace=True)

    return df

In [6]:
def fill_na_comprehensive(
    df: pd.DataFrame,
    embedding_cols: list[str] | None = None,
    string_fill: str = "",
    bool_fill: int | bool = 0,
    num_fill: float | int = 0.0,
) -> pd.DataFrame:

    df = df.copy()

    if embedding_cols is not None:
        for col in embedding_cols:
            if col not in df.columns:
                continue

            non_null = df[col].dropna()
            if non_null.empty:
                continue

            first_valid = non_null.iloc[0]
            emb_dim = len(first_valid)

            def _fill_emb(x):
                if isinstance(x, (np.ndarray, list)):
                    return x
                return np.zeros(emb_dim, dtype=float)

            df[col] = df[col].apply(_fill_emb)

    bool_cols = [c for c in df.columns if df[c].dtype == "bool"]

    bin_int_cols = []
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c].dtype):
            uniq = pd.Series(df[c].dropna().unique())
            if len(uniq) <= 2 and set(uniq.tolist()).issubset({0, 1}):
                bin_int_cols.append(c)

    df[bool_cols] = df[bool_cols].fillna(bool_fill)
    df[bin_int_cols] = df[bin_int_cols].fillna(bool_fill)

    obj_cols = [c for c in df.columns if df[c].dtype == "object" or str(df[c].dtype) == "string"]
    if embedding_cols is not None:
        obj_cols = [c for c in obj_cols if c not in embedding_cols]

    df[obj_cols] = df[obj_cols].fillna(string_fill)

    num_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c].dtype)
        and c not in bin_int_cols
    ]
    df[num_cols] = df[num_cols].fillna(num_fill)

    return df

In [ ]:

def data_fe(df: pd.DataFrame, model) -> pd.DataFrame:
    """
    Feature engineering with comprehensive dtype validation and accounting.
    """
    df = df.copy()
    
    # Track all transformations
    print(f"Initial shape: {df.shape}")
    print(f"Initial dtypes:\n{df.dtypes.value_counts()}\n")
    
    # Filter volume
    df = df[df['volume_total'] > 0]
    print(f"After volume filter: {df.shape}")
    
    # ===== DATETIME PROCESSING WITH VALIDATION =====
    df["snapshot_ts"] = pd.to_datetime(df["snapshot_ts"], errors="coerce")
    
    # Check for failed conversions
    failed_datetime = df["snapshot_ts"].isna().sum()
    if failed_datetime > 0:
        print(f"WARNING: {failed_datetime} datetime conversions failed")
    
    # Decompose datetime
    df["year"] = df["snapshot_ts"].dt.year
    df["month"] = df["snapshot_ts"].dt.month
    df["day"] = df["snapshot_ts"].dt.day
    df["hour"] = df["snapshot_ts"].dt.hour
    df["minute"] = df["snapshot_ts"].dt.minute
    df["second"] = df["snapshot_ts"].dt.second
    
    # Validate all datetime components were extracted
    datetime_components = ["year", "month", "day", "hour", "minute", "second"]
    for comp in datetime_components:
        null_count = df[comp].isna().sum()
        if null_count > 0:
            print(f"WARNING: {comp} has {null_count} null values after decomposition")
    
    # ===== COLUMN REMOVAL WITH TRACKING =====
    # Remove unnamed index columns
    unnamed_cols = [col for col in df.columns if col.startswith("Unnamed:")]
    if unnamed_cols:
        print(f"Dropping unnamed columns: {unnamed_cols}")
        df.drop(columns=unnamed_cols, inplace=True)
    
    # Remove all-null columns
    all_null_cols = df.columns[df.isna().all()].tolist()
    if all_null_cols:
        print(f"Dropping all-null columns: {all_null_cols}")
        df.drop(columns=all_null_cols, inplace=True)
    
    # Remove high-null columns
    threshold = 0.85
    null_ratio = df.isna().mean()
    cols_to_drop = null_ratio[null_ratio > threshold].index.tolist()
    if cols_to_drop:
        print(f"Dropping high-null columns (>{threshold}): {cols_to_drop}")
        df.drop(columns=cols_to_drop, inplace=True)
    
    # ===== TIME ACTIVE CALCULATION =====
    df = df.sort_values(["market_id", "snapshot_ts"])
    
    start = (
        df[df["active"]]
        .groupby("market_id")["snapshot_ts"]
        .min()
        .rename("start_time")
    )
    
    end = (
        df[df["closed"]]
        .groupby("market_id")["snapshot_ts"]
        .min()
        .rename("end_time")
    )
    
    result = pd.concat([start, end], axis=1).dropna(subset=["start_time", "end_time"])
    result["time_active_hours"] = (
        result["end_time"] - result["start_time"]
    ) / pd.Timedelta(hours=1)
    
    df["time_active"] = df["market_id"].map(result["time_active_hours"])
    
    # ===== DETECT AND HANDLE ALL STRING/OBJECT COLUMNS =====
    # Check for mixed types in columns
    mixed_dtype_cols = []
    for col in df.columns:
        inferred = infer_dtype(df[col], skipna=True)
        if inferred.startswith("mixed"):
            mixed_dtype_cols.append(col)
            print(f"WARNING: Column '{col}' has mixed dtype: {inferred}")
    
    # Identify ALL string/object/category columns
    object_cols = df.select_dtypes(include=['object']).columns.tolist()
    string_cols = df.select_dtypes(include=['string']).columns.tolist()
    category_cols = df.select_dtypes(include=['category']).columns.tolist()
    
    all_text_cols = list(set(object_cols + string_cols + category_cols))
    
    # Define known text columns for embedding
    text_cols_for_embedding = [
        "event_slug",
        "event_title",
        "market_slug",
        "question",
        "resolution_source",
        "event_description",
        "promotion_group_y",
    ]
    
    # Separate: columns to embed vs columns to handle differently
    cols_to_embed = [col for col in text_cols_for_embedding if col in df.columns]
    cols_to_handle = [col for col in all_text_cols if col not in text_cols_for_embedding]
    
    print(f"\nText columns to embed: {cols_to_embed}")
    print(f"Other text columns to handle: {cols_to_handle}")
    
    # ===== EMBED TEXT COLUMNS =====
    for col in cols_to_embed:
        df = embed_column(df, col, model)
        print(f"Embedded column: {col}")
    
    # ===== HANDLE REMAINING TEXT COLUMNS =====
    for col in cols_to_handle:
        if col == "market_id":
            # Special handling for market_id
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce').astype("Int64")
                failed_conversions = df[col].isna().sum()
                if failed_conversions > 0:
                    print(f"WARNING: {failed_conversions} market_id values failed numeric conversion")
            except Exception as e:
                print(f"ERROR converting market_id: {e}")
                df.drop(columns=col, inplace=True)
        else:
            # Drop other non-embedded text columns
            print(f"Dropping text column (not embedded): {col}")
            df.drop(columns=col, inplace=True)
    
    # ===== CONVERT DATETIME TO UNIX TIMESTAMP =====
    date_cols = ["snapshot_ts"]
    for col in date_cols:
        if col in df.columns:
            original_nulls = df[col].isna().sum()
            df[col] = df[col].astype("int64") // 10**9
            print(f"Converted {col} to unix timestamp ({original_nulls} nulls preserved)")
    
    # ===== CONVERT BOOLEAN COLUMNS =====
    bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
    for col in bool_cols:
        df[col] = df[col].astype("uint8")
    print(f"Converted {len(bool_cols)} boolean columns to uint8")
    
    # ===== FILL MISSING VALUES =====
    emb_cols = [f"{c}_emb" for c in cols_to_embed if f"{c}_emb" in df.columns]
    
    df = fill_na_comprehensive(
        df,
        embedding_cols=emb_cols,
        string_fill="",
        bool_fill=0,
        num_fill=0.0,
    )
    
    # ===== EXPLODE EMBEDDINGS =====
    df = explode_embeddings(df, emb_cols)
    
    # ===== FINAL VALIDATION: CHECK FOR REMAINING NON-NUMERIC TYPES =====
    final_object_cols = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    
    if final_object_cols:
        print(f"\nWARNING: Found {len(final_object_cols)} remaining non-numeric columns:")
        for col in final_object_cols:
            unique_types = df[col].apply(type).unique()
            sample_values = df[col].dropna().head(3).tolist()
            print(f"  - {col}: dtype={df[col].dtype}, types={unique_types}, samples={sample_values}")
            df.drop(columns=col, inplace=True)
            print(f"    -> Dropped")
    
    # ===== FINAL REPORT =====
    print(f"\n{'='*60}")
    print(f"Final shape: {df.shape}")
    print(f"Final dtypes:\n{df.dtypes.value_counts()}")
    print(f"Columns with nulls: {df.isna().any().sum()}")
    print(f"Total null values: {df.isna().sum().sum()}")
    
    # Verify all columns are numeric
    non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        print(f"\nERROR: Non-numeric columns remain: {non_numeric}")
    else:
        print(f"SUCCESS: All columns are numeric")
    
    return df


In [8]:
db_additional = pd.read_csv(f"{user_path}UEP-MachineLearning/data_model/db_additional.csv")
db_collected = pd.read_csv(f"{user_path}UEP-MachineLearning/data_model/db_collected.csv")

/var/folders/dk/zjwq8trx6cbbykf77jz61hnc0000gn/T/ipykernel_94008/3822338731.py:2: DtypeWarning: Columns (0: restricted, 1: promotion_group_x, 2: event_description, 3: resolved_by, 4: promotion_group_y) have mixed types. Specify dtype option on import or set low_memory=False.
  db_collected = pd.read_csv(f"{user_path}UEP-MachineLearning/data_model/db_collected.csv")


In [ ]:
db_additional = data_fe(db_additional, model=model)
db_collected = data_fe(db_collected, model=model)

KeyError: "None of [Index([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype='int64')] are in the [columns]"

In [ ]:
db_additional.to_csv(f"{user_path}UEP-MachineLearning/data_model/db_additional.csv")
db_collected.to_csv(f"{user_path}UEP-MachineLearning/data_model/db_collected.csv")